In [ ]:
import os
import json
import random
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Concatenate, Dropout, Attention
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback

DATA_PATH = "telugu_word_correction_pairs.json"
SAVE_DIR = "."
os.makedirs(SAVE_DIR, exist_ok=True)


EPOCHS = 10
BATCH_SIZE = 128
EMBEDDING_DIM = 64
LATENT_DIM = 128   
PATIENCE = 2


DEBUG = False         
DEBUG_SUBSET = 5000      
DEBUG_EPOCHS = 3        


TRAIN_SUBSET_SIZE = 30000   
VAL_SIZE = 10000
TEST_SIZE = 5000
USE_FULL_DATA = not DEBUG  

FINAL_MODEL_FILE = os.path.join(SAVE_DIR, "seq2seq_final.keras")
BEST_MODEL_FILE = os.path.join(SAVE_DIR, "seq2seq_best.keras")

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [2]:
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

if DEBUG:
    data = random.sample(data, min(DEBUG_SUBSET, len(data)))

total = len(data)
print(f" Loaded {total:,} pairs (Debug={DEBUG})")

indices = list(range(total))
random.shuffle(indices)

val_size = min(VAL_SIZE if not DEBUG else 1000, total // 5)  
test_size = min(TEST_SIZE if not DEBUG else 500, total // 10) 

if val_size + test_size >= total:
    val_size = max(1, total // 10)
    test_size = max(1, total // 10)

val_idx = set(indices[:val_size])
test_idx = set(indices[val_size:val_size + test_size])
train_idx = [i for i in indices if i not in val_idx and i not in test_idx]

train_pool = [data[i] for i in train_idx]
val_data = [data[i] for i in val_idx]
test_data = [data[i] for i in test_idx]

print(f"Train: {len(train_pool):,}, Val: {len(val_data):,}, Test: {len(test_data):,}")


 Loaded 1,251,815 pairs (Debug=False)
Train: 1,236,815, Val: 10,000, Test: 5,000


In [3]:
all_text = "".join([p["input"] + p["target"] for p in train_pool])
chars = sorted(set(all_text))

for tok in ['<sos>', '<eos>']:
    if tok not in chars:
        chars.append(tok)

char2idx = {c: i+1 for i, c in enumerate(chars)}  
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx) + 1

max_len = max(max(len(p["input"]), len(p["target"])) + 2 for p in train_pool)
print(f"Unique chars: {len(char2idx)}, Max length: {max_len}, Vocab size: {vocab_size}")

def encode_word_with_tokens(word):
    return [char2idx['<sos>']] + [char2idx.get(ch, 0) for ch in word] + [char2idx['<eos>']]


Unique chars: 102, Max length: 129, Vocab size: 103


In [4]:
def make_generator(data_split, batch_size=BATCH_SIZE):
    def gen():
        Xb, Db, Yb = [], [], []
        while True:
            random.shuffle(data_split)
            for pair in data_split:
                enc = encode_word_with_tokens(pair['input'])
                dec_in = [0] + encode_word_with_tokens(pair['target'])[:-1]
                dec_out = encode_word_with_tokens(pair['target'])
                Xb.append(enc)
                Db.append(dec_in)
                Yb.append(dec_out)
                if len(Xb) == batch_size:
                    Xp = pad_sequences(Xb, maxlen=max_len, padding='post')
                    Dp = pad_sequences(Db, maxlen=max_len, padding='post')
                    Yp = pad_sequences(Yb, maxlen=max_len, padding='post')
                    yield (Xp.astype(np.int32), Dp.astype(np.int32)), np.expand_dims(Yp, -1).astype(np.int32)
                    Xb, Db, Yb = [], [], []
    return gen

output_signature = (
    (tf.TensorSpec(shape=(None, max_len), dtype=tf.int32),
     tf.TensorSpec(shape=(None, max_len), dtype=tf.int32)),
    tf.TensorSpec(shape=(None, max_len, 1), dtype=tf.int32)
)


In [5]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Bidirectional, Attention, Concatenate, Dropout, TimeDistributed, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

encoder_inputs = Input(shape=(max_len,), name='encoder_inputs')
enc_emb = Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True, name='enc_emb')(encoder_inputs)

encoder_outputs, forward_h, forward_c, backward_h, backward_c = Bidirectional(
    LSTM(LATENT_DIM, return_sequences=True, return_state=True, dropout=0.15, recurrent_dropout=0.15),
    name='bidirectional_encoder'
)(enc_emb)

state_h = Concatenate(name='enc_state_h')([forward_h, backward_h])
state_c = Concatenate(name='enc_state_c')([forward_c, backward_c])

decoder_inputs = Input(shape=(max_len,), name='decoder_inputs')
dec_emb = Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True, name='dec_emb')(decoder_inputs)

decoder_lstm = LSTM(LATENT_DIM * 2, return_sequences=True, return_state=True, dropout=0.15, recurrent_dropout=0.15, name='decoder_lstm')
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

attention = Attention(name='luong_attention')
attn_out = attention([decoder_outputs, encoder_outputs])

concat = Concatenate(axis=-1, name='context_concat')([decoder_outputs, attn_out])
concat = LayerNormalization()(concat)
concat = Dropout(0.2, name='context_dropout')(concat)

decoder_dense = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')
decoder_outputs = decoder_dense(concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

optimizer = Adam(learning_rate=3e-4, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()




Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 encoder_inputs (InputLayer  [(None, 129)]                0         []                            
 )                                                                                                
                                                                                                  
 enc_emb (Embedding)         (None, 129, 64)              6592      ['encoder_inputs[0][0]']      
                                                                                                  
 decoder_inputs (InputLayer  [(None, 129)]                0         []                            
 )                                                                                                
                                                                                            

In [6]:
class TqdmProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self, steps_per_epoch):
        super().__init__()
        self.steps_per_epoch = int(steps_per_epoch)
        self.pbar = None

    def on_epoch_begin(self, epoch, logs=None):
        self.pbar = tqdm(total=self.steps_per_epoch, desc=f"Epoch {epoch+1}", unit="batch", ncols=100)

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        loss = logs.get("loss", 0)
        acc = logs.get("accuracy", 0)
        self.pbar.set_postfix({"loss": f"{loss:.3f}", "acc": f"{acc:.3f}"})
        self.pbar.update(1)

    def on_epoch_end(self, epoch, logs=None):
        if self.pbar:
            self.pbar.close()
        val_loss = logs.get("val_loss", 0)
        val_acc = logs.get("val_accuracy", 0)
        print(f"\n Epoch {epoch+1} complete | val_loss={val_loss:.3f} | val_acc={val_acc:.3f}\n")


In [7]:
if os.path.exists(FINAL_MODEL_FILE):
    print(" Found saved model. Loading existing model...")
    model = tf.keras.models.load_model(FINAL_MODEL_FILE)
else:
    print(" Starting training from scratch...")
    val_dataset = tf.data.Dataset.from_generator(make_generator(val_data, batch_size=BATCH_SIZE), output_signature=output_signature)

    if DEBUG:
        steps_per_epoch = max(1, len(train_pool) // BATCH_SIZE)
        total_epochs = DEBUG_EPOCHS
    else:
        steps_per_epoch = max(1, TRAIN_SUBSET_SIZE // BATCH_SIZE)
        total_epochs = EPOCHS

    early_stop = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
    checkpoint = ModelCheckpoint(BEST_MODEL_FILE, monitor='val_loss', save_best_only=True, verbose=1)

    for epoch in range(total_epochs):
        print(f"\n=== Epoch {epoch+1}/{total_epochs} ===")
        if DEBUG:
            subset = random.sample(train_pool, min(len(train_pool), TRAIN_SUBSET_SIZE))
        else:
            subset = random.sample(train_pool, min(len(train_pool), TRAIN_SUBSET_SIZE))

        train_dataset = tf.data.Dataset.from_generator(make_generator(subset, batch_size=BATCH_SIZE), output_signature=output_signature)

        progress_cb = TqdmProgressCallback(steps_per_epoch)
        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            steps_per_epoch=steps_per_epoch,
            validation_steps=max(1, len(val_data)//BATCH_SIZE),
            epochs=1,
            callbacks=[early_stop, checkpoint, progress_cb],
            verbose=0
        )

        model.save(FINAL_MODEL_FILE)
        print(f" Saved model to {FINAL_MODEL_FILE}")

        if early_stop.stopped_epoch:
            print(" Early stopping triggered. Ending training.")
            break

    print("Training finished (or skipped if model loaded).")


 Found saved model. Loading existing model...


In [8]:
def beam_search_predict(model, input_word, beam_width=5, length_penalty=0.7, max_repeat=3):
    """
    Beam search decoder with length penalty and early stopping for Telugu Seq2Seq.
    Args:
        model: trained seq2seq model
        input_word: noisy input string
        beam_width: number of beams to keep
        length_penalty: controls favoring longer sequences (0.6–1.0 works well)
        max_repeat: max allowed single-character repetition before trimming
    """
    enc_seq = pad_sequences([encode_word_with_tokens(input_word)], maxlen=max_len, padding='post')

    sequences = [([char2idx['<sos>']], 0.0)]
    completed = []

    for _ in range(max_len):
        all_candidates = []
        stop_counter = 0 

        for seq, score in sequences:
            if seq[-1] == char2idx['<eos>']:
                completed.append((seq, score))
                stop_counter += 1
                continue

            dec_seq = pad_sequences([seq], maxlen=max_len, padding='post')
            preds = model.predict([enc_seq, dec_seq], verbose=0)
            probs = preds[0, len(seq)-1]

            probs = np.nan_to_num(probs, nan=1e-9, posinf=1e-9, neginf=1e-9)

            top_idx = np.argsort(probs)[-beam_width:]
            for i in top_idx:
                candidate = seq + [int(i)]
                new_score = (score + np.log(probs[int(i)] + 1e-9)) / (len(candidate) ** length_penalty)
                all_candidates.append((candidate, new_score))

        if stop_counter == beam_width:
            break

        sequences = sorted(all_candidates, key=lambda t: t[1], reverse=True)[:beam_width]

    best_seq = sorted(completed or sequences, key=lambda t: t[1], reverse=True)[0][0]

    decoded = ''.join([
        idx2char.get(i, '') for i in best_seq 
        if i not in (0, char2idx['<sos>'], char2idx['<eos>'])
    ])

    if len(decoded) > max_repeat and len(set(decoded)) == 1:
        decoded = decoded[0]

    return decoded


In [9]:
class HybridSpellChecker:
    def __init__(self, vocabulary, seq2seq_model, char2idx, idx2char, max_len, prefix_len=2):
        self.vocab = sorted(list(set(vocabulary)))
        self.model = seq2seq_model
        self.char2idx = char2idx
        self.idx2char = idx2char
        self.max_len = max_len
        self.prefix_len = prefix_len
        self.groups = {}
        for w in self.vocab:
            key = w[:prefix_len]
            self.groups.setdefault(key, []).append(w)

    def levenshtein(self, a, b):
        m, n = len(a), len(b)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1): dp[i][0] = i
        for j in range(n+1): dp[0][j] = j
        for i in range(1, m+1):
            ai = a[i-1]
            for j in range(1, n+1):
                cost = 0 if ai == b[j-1] else 1
                dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
        return dp[m][n]

    def seq2seq_confidence(self, enc_seq, candidate):
        dec_in = [self.char2idx['<sos>']] + [self.char2idx.get(c, 0) for c in candidate]
        dec_in_p = pad_sequences([dec_in], maxlen=self.max_len, padding='post')
        preds = self.model.predict([enc_seq, dec_in_p], verbose=0)
        conf = float(np.mean(np.max(preds[0], axis=-1)))
        return conf

    def candidates_by_levenshtein(self, noisy_word, max_suggestions=5, max_distance=2):
        key = noisy_word[:self.prefix_len]
        candidates = self.groups.get(key, [])
        close = []
        for w in candidates:
            d = self.levenshtein(noisy_word, w)
            if d <= max_distance:
                close.append((w, d))
        close.sort(key=lambda x: x[1])
        return [w for w, _ in close[:max_suggestions]]

    def correct_word(self, noisy_word, max_suggestions=5, max_distance=2):
        cand = self.candidates_by_levenshtein(noisy_word, max_suggestions, max_distance)
        enc_seq = pad_sequences([encode_word_with_tokens(noisy_word)], maxlen=self.max_len, padding='post')
        if not cand:
            return beam_search_predict(self.model, noisy_word)
        best_word = None
        best_conf = -1.0
        for c in cand:
            conf = self.seq2seq_confidence(enc_seq, c)
            if conf > best_conf:
                best_conf = conf
                best_word = c
        return best_word if best_word else beam_search_predict(self.model, noisy_word)


In [ ]:
from tqdm import tqdm

vocab_words = [p['target'] for p in train_pool]  
hybrid = HybridSpellChecker(vocab_words, model, char2idx, idx2char, max_len)

print(f" HybridSpellChecker initialized with {len(vocab_words):,} vocabulary words.\n")

num_samples = min(5, len(test_data))
print(f" Evaluating {num_samples} random samples...\n")
for sample in random.sample(test_data, num_samples):
    noisy = sample['input']
    target = sample['target']
    greedy_pred = beam_search_predict(model, noisy, beam_width=1)
    beam_pred = beam_search_predict(model, noisy, beam_width=3)
    hybrid_pred = hybrid.correct_word(noisy, max_suggestions=3, max_distance=1)
    print(f"Noisy : {noisy}")
    print(f"Target: {target}")
    print(f"Greedy Seq2Seq → {greedy_pred}")
    print(f"Beam Search (width=3) → {beam_pred}")
    print(f"Hybrid (Levenshtein + DL) → {hybrid_pred}")
    print("-" * 60)




 HybridSpellChecker initialized with 1,236,815 vocabulary words.

 Evaluating 5 random samples...

Noisy : ఖరంాప్
Target: ఖరాప్
Greedy Seq2Seq → ఖరాంప్
Beam Search (width=3) → ఖరాంప్లోరింగారుకిఃంపీఏరేఖ్పరైప్పరైత్పరేషప్తరీపైఖరేఖత్పరేఖంత్పరీఖేఖకర్పైఖతేషరీపఖేఖర్తంపైకరేఖీఖంత్రేఖపైకరీఖంత్రేఖంపైర్తపేఖరీ
Hybrid (Levenshtein + DL) → ఖరాప్
------------------------------------------------------------
Noisy : పూర్తశచేశాడు
Target: పూర్తిచేశాడు
Greedy Seq2Seq → పూర్తశేశాడు
Beam Search (width=3) → సూర్శతేశాడు
Hybrid (Levenshtein + DL) → పూర్తిచేశాడు
------------------------------------------------------------
Noisy : ప్రణత్నించాయి
Target: ప్రయత్నించాయి
Greedy Seq2Seq → ప్రత్ణనించాయి
Beam Search (width=3) → ప్రత్ణనించాయి
Hybrid (Levenshtein + DL) → ప్రయత్నించాయి
------------------------------------------------------------
Noisy : వియమ
Target: వియమ
Greedy Seq2Seq → వియమ
Beam Search (width=3) → ఈవియమాలికుందికియీఏూఏికీర్పైరిక్పీకరేషత్పైకర్తీఏరేక్పరణేఫీకర్తేఖరీఖక్తేఏరీఖకేఖతీఏరేఖకీఖంతేర్ఐఏకీఖేర్తైకీరేఖ్తూ

In [ ]:
#takes lots of time_ ignore this cell
N = min(50, len(test_data))   
print(f"\n Evaluating hybrid accuracy on {N} samples (this may take time)...\n")

correct_hybrid = correct_beam = 0
for s in tqdm(random.sample(test_data, N), desc="Hybrid Eval"):
    noisy = s['input']; target = s['target']
    if hybrid.correct_word(noisy, max_suggestions=3, max_distance=1) == target:
        correct_hybrid += 1
    if beam_search_predict(model, noisy, beam_width=3) == target:
        correct_beam += 1

print(f"\n Hybrid Accuracy (N={N}): {correct_hybrid/N*100:.2f}%")
print(f" Beam Search Accuracy (N={N}): {correct_beam/N*100:.2f}%")

In [ ]:
from tqdm import tqdm

vocab_words = [p['target'] for p in data]  
hybrid = HybridSpellChecker(vocab_words, model, char2idx, idx2char, max_len)

print(f" HybridSpellChecker initialized with {len(vocab_words):,} vocabulary words.\n")

test_word = "భారదేతశం"  

greedy_pred = beam_search_predict(model, test_word, beam_width=1)
beam_pred = beam_search_predict(model, test_word, beam_width=3)
hybrid_pred = hybrid.correct_word(test_word, max_suggestions=3, max_distance=1)

print("Single Word Correction Test\n")
print(f"Noisy Input : {test_word}")
print(f"Greedy Seq2Seq → {greedy_pred}")
print(f"Beam Search (width=3) → {beam_pred}")
print(f"Hybrid (Levenshtein + DL) → {hybrid_pred}")
print("-" * 70)


✅ HybridSpellChecker initialized with 1,251,815 vocabulary words.

🔍 Single Word Correction Test

Noisy Input : భారదేతశం
Greedy Seq2Seq → భారదేశంత
Beam Search (width=3) → భారదేశంతేనందికీతేంభేరికితంపేరీకప్తరైపక్తరేషతీపేకరత్పేరీతేకరంతేశ్కరీతేశంకరేత్పీకరేశంభతేరకీపేశంభరేత్పకీషేరత్పైరకే
Hybrid (Levenshtein + DL) → దారభేశంతేదిరంతోభేశికీతిరేపకైరపిశీకరతేషపర్తేకరపీతేటరక్తేశీకరతేష్రీతేఖకరీతేశంభరకేశీపతరేషంతేకరపీతేశంకరత్భేశీకరపేషతంతరేశీపకరేంతూరకే
----------------------------------------------------------------------
